<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/01_timer/01_timer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Install GNU Fortran and NVIDIA HPC SDK (optional for C-only examples; can take ~30 min)

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download, not needed for C-only examples).
# Uncomment the following lines to install it.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi

import glob, os
nvhpc_bins = sorted(glob.glob('/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin'), key=lambda p: tuple(int(x) for x in p.split('/Linux_x86_64/')[1].split('/')[0].replace('-', '.').split('.')), reverse=True)
if nvhpc_bins:
    nvhpc_bin = nvhpc_bins[0]
    current_path = os.environ.get('PATH', '')
    if nvhpc_bin not in current_path.split(':'):
        os.environ['PATH'] = nvhpc_bin + (':' + current_path if current_path else '')
    print('Using NVIDIA HPC SDK:', nvhpc_bin)
else:
    print('NVIDIA HPC SDK not found (optional; not needed for C-only examples).')


Clone the repository and change to the `01_timer` directory.

In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd HPC-Programming/Tuning/sample_code/01_timer
!ls


## 01_timer: By-hand Timers and gprof
* Author:   Yukihiro Ota (yota@rist.or.jp)
* Last update: 7th Aug., 2026

### Purpose
This sample demonstrates three basic techniques for measuring the performance of a program, which are the first steps of any tuning work:

1. **Elapsed (wall-clock) time measurement** with a by-hand timer inserted in the source code
2. **CPU time measurement** with a by-hand timer
3. **Profiling with `gprof`** to find hotspots without modifying the source code

The sample program (`main.c` / `main.f90`) calls three subroutines (`sub1`, `sub2`, `sub3`) with different call counts and workloads. By timing and profiling them, you will learn how to identify the hotspot of a program.

### Directory layout
```
01_timer/
├── src/            # Source code and Makefiles
│   ├── c/          # C version (default)
│   ├── fortran/    # Fortran version
│   ├── fortran_c/  # Optional: Fortran with a C timer
│   └── cpp/        # Optional: C++ with std::chrono
└── tests/          # Job scripts (run.sh) for each language
    ├── c/
    ├── fortran/
    ├── fortran_c/
    └── cpp/
```

Choose either C or Fortran. The examples below use C; for Fortran, replace `src/c` and `tests/c` with `src/fortran` and `tests/fortran`, and edit `FFLAGS` instead of `CFLAGS` in the Makefile.

The timer mode is selected by the compiler flags in the `Makefile` (`src/<lang>/Makefile`):

| Mode | Flag setting in Makefile |
|---|---|
| Elapsed time (wall clock) | `-DUSE_ELP_TIMER` (default) |
| CPU time | `-DUSE_CPU_TIMER` |
| gprof profiling | `-pg` (no `-DUSE_*_TIMER`) |

The code has been verified with GNU compilers (11.4.0) on x86-64 systems.
If linking fails, try `LIB=-lm -lrt` in the Makefile.

### Exercise steps

#### Step 1: Measure elapsed (wall-clock) time
1. Move to the source directory and build. The default Makefile already sets `-DUSE_ELP_TIMER`:

In [ ]:
%%bash
cd src/c
make

2. Move to the test directory and run the job script:

In [ ]:
%%bash
cd tests/c
bash run.sh

3. Check `outfile`. The elapsed time of each timed section is printed as:
   ```
   Elapsed time (sec) = ...
   ```

In [ ]:
%%bash
cat tests/c/outfile

4. Compare the elapsed times of the two timed loops (routine 1 calling `sub1`, and routine 2 calling `sub2`) and consider which one is more expensive and why.

#### Step 2: Measure CPU time
1. Edit `src/c/Makefile`: comment out the `-DUSE_ELP_TIMER` line and enable the `-DUSE_CPU_TIMER` line:
   ```makefile
   ## Use of timer for Elapsed time
   #CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_ELP_TIMER
   ## Use of timer for CPU time
   CFLAGS=-g -Wall -O0 -std=gnu99 -DUSE_CPU_TIMER
   ```
   For Fortran (`src/fortran/Makefile`), edit `FFLAGS` instead:
   ```makefile
   ## Use of timer (elapsed time)
   #FFLAGS=-g -Wall -cpp -O0 -DUSE_ELP_TIMER
   ## Use of timer for CPU time
   FFLAGS=-g -Wall -cpp -O0 -DUSE_CPU_TIMER
   ```
2. Rebuild in `src/c`, then move to the test directory and rerun:

In [ ]:
%%bash
cd src/c
make veryclean && make
cd ../../tests/c
bash run.sh

3. Check `outfile` for lines such as:
   ```
   CPU time (sec)     = ...
   ```

In [ ]:
%%bash
cat tests/c/outfile

4. Compare CPU time with the elapsed time from Step 1. Note that the resolution of the CPU timer is coarser; you may have to enlarge the array size (`nn` in `main.c`) or the loop counts to obtain meaningful values.

#### Step 3: Profile with gprof
1. Edit `src/c/Makefile` again: disable the `-DUSE_*_TIMER` flags and enable the `-pg` line:
   ```makefile
   ## Use of gprof
   CFLAGS=-pg -g -Wall -O0 -std=gnu99
   ```
   For Fortran (`src/fortran/Makefile`), edit `FFLAGS` instead:
   ```makefile
   ## Use of gprof
   FFLAGS=-pg -g -Wall -cpp -O0
   ```
2. Rebuild:

In [ ]:
%%bash
cd src/c
make veryclean && make

3. In `tests/c/run.sh`, uncomment the gprof lines:
   ```bash
   sleep 10s
   gprof $EXE > prof.out
   ```
4. Move to the test directory and run the job script:

In [ ]:
%%bash
cd tests/c
bash run.sh

5. The `gprof` result is summarized in `prof.out`. Examine the flat profile and the call graph to find the functions corresponding to the hotspot, and confirm that the result is consistent with the by-hand timer measurements.

In [ ]:
%%bash
cat tests/c/prof.out

> **Note:** The profiling data file `gmon.out` is created in the directory where the program *runs*, i.e., `tests/c/` when using `run.sh` — not in `src/c/`. This is why `run.sh` invokes `gprof` there. To start over from a clean state, remove the generated files:
> ```
> $ cd tests/c
> $ rm -f gmon.out prof.out outfile
> ```

### Questions to consider
1. Which function is the hotspot, and how do the call counts of `sub1`, `sub2`, and `sub3` explain it?
2. When do elapsed time and CPU time differ, and which one should you use for tuning?
3. What are the pros and cons of by-hand timers vs. `gprof`?